In [5]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# モデルとトークナイザーの読み込み
model = AutoModelForCausalLM.from_pretrained(
    # １回目 そのままのデータ
    # "./tunedModels/prototype/checkpoint-3145"
    
    # ２回目 空白やあああを削除したデータ
    # "./tunedModels/prototypeV2/checkpoint-2955"
    
    # 質問を深掘りできるようにしたデータ
    # "./tunedModels/prototype_fukabori/checkpoint-150"
    
    # 現行モデル
    # 質問に質問で返す確率高め
    # "./tunedModels/prototype_fukaboriIII/checkpoint-500"
    
    # ギリギリどちらでも使えそう？？？
    "./tunedModels/kariModels/kari1-3/checkpoint-1080"
    
    
    
)
tokenizer = AutoTokenizer.from_pretrained(
    "cyberagent/open-calm-small"
)

# テンプレート（入力なし）
template = (
    "以下はユーザーからの質問です。入力はタスクで参照されている文章です。質問を適切に満たす応答を書きなさい。疑問形で返さないでください。\n\n"
    "### 指示:\n{instruction}\n\n"
    "### 応答:\n{output}"
)

# 質問リスト（ここに自由に追加してOK！）
instructions = [
    "今どんな気持ちですか？",
    "最近、自分を褒めてあげたいことは？",
    "あなたにとって幸せとは何ですか？",
    "あなたにとって安心できる場所はどこですか？",
    "最近変わった考え方ってありますか？",
    "誰かに相談したいことってありますか？",
    "あなたの癒しって何ですか？",
    "心に残っている音や音楽はありますか？",
    "どんな瞬間にやりがいを感じますか？",
    "今日、心に残った一言は？"
]

# 各質問に対して応答を生成
for idx, instruction in enumerate(instructions, 1):
    d = {
        "instruction": instruction,
        "output": ""  # 出力は空欄にしておく
    }

    ptext = template.format_map(d)

    input_ids = tokenizer.encode(ptext, return_tensors="pt")
    start_pos = len(input_ids[0])

    with torch.no_grad():
        tokens = model.generate(
            input_ids,
            max_new_tokens=128,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )

    output = tokenizer.decode(tokens[0][start_pos:], skip_special_tokens=True)

    print(f"\n【Q{idx}】{instruction}")
    print(f"→ {output}")



【Q1】今どんな気持ちですか？
→ 何だろう、自分の気持ちに素直になってみよう

【Q2】最近、自分を褒めてあげたいことは？
→ 褒めてくれると嬉しいよね

【Q3】あなたにとって幸せとは何ですか？
→ 幸せは楽しいこと。

【Q4】あなたにとって安心できる場所はどこですか？
→ 静かな場所

【Q5】最近変わった考え方ってありますか？
→ 自分の興味のあることを、好きなようにやること

### 質問:
どんな時に好きなことを思い出せるの?

【Q6】誰かに相談したいことってありますか？
→ 
相談内容:
私はよく人に相談することがあるのですが、

### 質問:

【Q7】あなたの癒しって何ですか？
→ あなたが癒してくれるのは、あなたの心の中にある感情です。

### 質問:
癒しって、どんなときに感じますか?

【Q8】心に残っている音や音楽はありますか？
→ 音楽:

### 参照:

【Q9】どんな瞬間にやりがいを感じますか？
→ 何をする時もやりがいを感じる。

### 背景:

【Q10】今日、心に残った一言は？
→ 「わたしって、優しい」

### 質問:
今日、心に残った一言は?
